In [10]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from pydantic import BaseModel
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [11]:
load_dotenv()

True

In [12]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [26]:
Mood = Literal["happy", "sad", "neutral"]
class MoodState(TypedDict):
    userInput: str
    mood: Mood
    reply: str


def detectMood(state: MoodState) -> MoodState:
    prompt = f"""
    You are a mood detection assistant. Based on the user's input, determine their mood as either happy, sad, or neutral. Then, provide an appropriate reply to the user.

    User Input: {state['userInput']}
"""
    text = llm.invoke(prompt)

    if "happy" in text.content:
        mood: Mood = "happy"
    elif "sad" in text.content:
        mood: Mood = "sad"
    else:
        mood: Mood = "neutral"
    
    return {"mood" : mood}

In [27]:
graph = StateGraph(MoodState)

graph.add_node("detectmood", detectMood)
graph.add_edge(START, "detectmood")
graph.add_edge("detectmood", END)

workflow = graph.compile()

In [32]:
result = workflow.invoke({"userInput": "I am dead!"})
print(result["mood"])

sad
